# Ingestion to Database

## Imports

In [28]:
import psycopg2
import psycopg2.extensions
import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine

## fetch Postgres credentials

In [29]:
load_dotenv(dotenv_path='../.env')
db_name = os.getenv('DB_NAME')
db_user = os.getenv('DB_USER')
db_password = os.getenv('DB_PASSWORD')
db_host = os.getenv('DB_HOST')
db_port = os.getenv('DB_PORT')

In [30]:
# Create SQLAlchemy engine for PostgreSQL
engine = create_engine(f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")

## Create Database if it doesn't exist

In [31]:
try:
    conn = psycopg2.connect(dbname='postgres', user=db_user, password=db_password, host=db_host, port=db_port)
    conn.set_isolation_level(psycopg2.extensions.ISOLATION_LEVEL_AUTOCOMMIT)
    cur = conn.cursor()
    cur.execute(f"SELECT 1 FROM pg_database WHERE datname = '{db_name}'")
    exists = cur.fetchone()
    if not exists:
        cur.execute(f'CREATE DATABASE {db_name}')
        print(f'Database: {db_name} created.')
    else:
        print(f'Database: {db_name} already exists.')
    cur.close()
    conn.close()
except Exception as e:
    print('Error:', e)

Database: shop_project_database already exists.


## Create tables if they don't exist

In [32]:
try:
    conn = psycopg2.connect(dbname=db_name, user=db_user, password=db_password, host=db_host, port=db_port)
    cur = conn.cursor()
    # Create customers table
    cur.execute("""CREATE TABLE IF NOT EXISTS customers (
        customer_id VARCHAR PRIMARY KEY,
        first_name VARCHAR,
        last_name VARCHAR,
        email VARCHAR,
        city VARCHAR,
        region VARCHAR,
        country VARCHAR,
        signup_date DATE,
        age INTEGER,
        is_subscribed BOOLEAN,
        loyalty_tier VARCHAR
    )""")
    # Create orders table
    cur.execute("""CREATE TABLE IF NOT EXISTS orders (
        order_id VARCHAR PRIMARY KEY,
        customer_id VARCHAR REFERENCES customers(customer_id),
        product_id VARCHAR REFERENCES products(product_id),
        order_date DATE,
        quantity INTEGER,
        unit_price NUMERIC,
        total_amount NUMERIC,
        order_status VARCHAR,
        payment_method VARCHAR
    )""")
    # Create products table
    cur.execute("""CREATE TABLE IF NOT EXISTS products (
        product_id VARCHAR PRIMARY KEY,
        product_name VARCHAR,
        category VARCHAR,
        price NUMERIC,
        cost NUMERIC,
        stock_quantity INTEGER,
        supplier VARCHAR
    )""")
    conn.commit()
    cur.close()
    conn.close()
    print('Tables created or already exist.')
except Exception as e:
    print('Error:', e)

Tables created or already exist.


## Ingest CSV Data into Database
This section reads the CSV files and inserts their data into the corresponding tables: customers, products, and orders.

In [33]:
# Read CSVs
customers_df = pd.read_csv('../CSVs/customers.csv')
products_df = pd.read_csv('../CSVs/products.csv')
orders_df = pd.read_csv('../CSVs/orders.csv')

print('CSV files loaded:')
print('Customers:', customers_df.shape)
print('Products:', products_df.shape)
print('Orders:', orders_df.shape)

CSV files loaded:
Customers: (1000, 11)
Products: (400, 7)
Orders: (2000, 9)


## Insert DF data to respective table

In [ ]:
# Insert data into customers, products, and orders tables using SQLAlchemy

def insert_dataframe_to_table(df, table_name, engine, columns=None):
    """
    Inserts all rows from a pandas DataFrame into a specified PostgreSQL table.
    Parameters:
        df (pd.DataFrame): The DataFrame containing the data to insert.
        table_name (str): The name of the target table in the database.
        engine: The SQLAlchemy engine object.
        columns (list, optional): List of column names to insert. If None, uses all DataFrame columns.
    """
    # Use pandas to_sql() for insertion
    try:
        df.to_sql(table_name, engine, if_exists='append', index=False)
        print(f'Data inserted into {table_name}.')
    except Exception as e:
        if 'duplicate key value violates unique constraint' in str(e):
            print(f"Duplicate keys found in {table_name}. Skipping duplicates.")
        else:
            print(f"Error inserting into {table_name}: {e}")

try:
    insert_dataframe_to_table(customers_df, 'customers', engine)
    insert_dataframe_to_table(products_df, 'products', engine)
    insert_dataframe_to_table(orders_df, 'orders', engine)
except Exception as e:
    print('Error during data ingestion:', e)

Duplicate keys found in customers. Skipping duplicates.
Duplicate keys found in products. Skipping duplicates.
Duplicate keys found in orders. Skipping duplicates.


## Display preview of each table

In [35]:
def show_table_head(table_name):
    try:
        df = pd.read_sql_query(f'SELECT * FROM {table_name} LIMIT 5', engine)
        print(f'First 5 rows of {table_name}:')
        display(df)
    except Exception as e:
        print(f'Error displaying {table_name}:', e)

show_table_head('customers')
show_table_head('products')
show_table_head('orders')

First 5 rows of customers:


,customer_id,first_name,last_name,email,city,region,country,signup_date,age,is_subscribed,loyalty_tier
0,CUST-0001,James,Brown,james.brown@gmail.com,London,England,United Kingdom,2022-01-31,65,True,Bronze
1,CUST-0002,Emily,Campbell,emily.campbell@outlook.com,Birmingham,England,United Kingdom,2023-09-25,55,True,Bronze
2,CUST-0003,Arthur,Roberts,arthur.roberts@icloud.com,London,England,United Kingdom,2021-03-25,53,False,Bronze
3,CUST-0004,Freddie,Murphy,freddie.murphy@outlook.com,London,England,United Kingdom,2020-10-15,66,False,Bronze
4,CUST-0005,Leo,Stewart,leo.stewart@hotmail.com,Glasgow,Scotland,United Kingdom,2020-01-15,24,False,Bronze


First 5 rows of products:


,product_id,product_name,category,price,cost,stock_quantity,supplier
0,PRD_001,Smart Webcam,Electronics,125.21,91.76,322,TD SYNNEX
1,PRD_002,Smart Speaker,Electronics,336.32,191.26,469,TD SYNNEX
2,PRD_003,Rechargeable Speaker,Electronics,150.81,79.02,574,Tech Data
3,PRD_004,Wireless Router,Electronics,149.79,93.68,597,Exertis
4,PRD_005,4K Monitor,Electronics,196.94,147.55,141,Tech Data


First 5 rows of orders:


,order_id,customer_id,product_id,order_date,quantity,unit_price,total_amount,order_status,payment_method
0,ORD-000001,CUST-0251,PRD_328,2023-02-21,1,27.96,27.96,Completed,Credit Card
1,ORD-000002,CUST-0033,PRD_347,2023-06-28,5,51.74,258.70,Completed,Debit Card
2,ORD-000003,CUST-0734,PRD_120,2023-02-24,5,96.94,484.70,Completed,Google Pay
3,ORD-000004,CUST-0829,PRD_215,2025-07-08,2,35.22,70.44,Pending,Debit Card
4,ORD-000005,CUST-0221,PRD_358,2024-11-27,4,10.92,43.68,Returned,PayPal
